In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
from pathlib import Path
import uuid

In [6]:
print(os.getcwd())
df = pd.read_csv('../../data/en/metadata.csv')
print(df.shape)

cv_path = os.path.join(os.getcwd(), '../../data', 'en', 'clips')

cv_list = os.listdir(cv_path)

df = df[df['path'].isin(cv_list)]
print(df.shape)

/root/master_thesis/thesis_multi_speaker_asr/src/multi_speaker_asr
(45737, 11)
(45737, 11)


In [8]:
print(df.columns)
df = df.drop(df.columns[[0]], axis=1)
print(df.columns)

Index(['Unnamed: 0', 'client_id', 'path', 'sentence_id', 'sentence', 'age',
       'gender', 'accents', 'locale', 'clip', 'duration[ms]'],
      dtype='str')
Index(['client_id', 'path', 'sentence_id', 'sentence', 'age', 'gender',
       'accents', 'locale', 'clip', 'duration[ms]'],
      dtype='str')


In [9]:
print(df.shape)

df['uuid'] = [str(uuid.uuid4()) for _ in range(len(df))]
df.to_csv('metadata.csv', index=False)

(45737, 10)


In [ ]:
import matplotlib.pyplot as plt

age_order = [
    'teens', 'twenties', 'thirties', 'fourties',
    'fifties', 'sixties', 'seventies',
    'eighties', 'nineties'
]
age_dist = df['age'].value_counts(normalize=True).reindex(age_order)
gender_dist = df['gender'].value_counts(normalize=True)

top_accents = df['accents'].value_counts().head(10)
accents_dist = top_accents / top_accents.sum()

# --- Plot grid ---
fig, axes = plt.subplots(3, 1, figsize=(8, 12))

# Age
axes[0].bar(range(len(age_dist)), age_dist.values)
axes[0].set_title("Age distribution")
axes[0].set_xlabel("Age bucket")
axes[0].set_ylabel("Proportion")
axes[0].set_xticks(range(len(age_dist)))
axes[0].set_xticklabels(age_dist.index, rotation=45)

# Gender
axes[1].bar(gender_dist.index, gender_dist.values)
axes[1].set_title("Gender distribution")
axes[1].set_xlabel("Gender")
axes[1].set_xticks(range(len(gender_dist)))
axes[1].set_xticklabels(gender_dist.index, rotation=45)
axes[1].set_ylabel("Proportion")

# Accent (top 10)
axes[2].bar(range(len(accents_dist)), accents_dist.values)
axes[2].set_title("Top 10 accents (normalized)")
axes[2].set_xlabel("Accent")
axes[2].set_ylabel("Proportion")
axes[2].set_xticks(range(len(accents_dist)))
axes[2].set_xticklabels(accents_dist.index, rotation=90)

plt.tight_layout()
plt.show()

In [1]:
import pandas as pd
import uuid
import os
import shutil
import datacollective
from dotenv import load_dotenv

In [2]:
df_duration = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/commonvoice-en/cv-corpus-25.0-2026-03-09/en/clip_durations.tsv', sep='\t')
df_test = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/commonvoice-en/cv-corpus-25.0-2026-03-09/en/test.tsv', sep='\t')

In [3]:
cols = ['client_id', 'path', 'sentence_id', 'sentence',
        'age', 'gender', 'accents']

for col in cols:
    missing = df_test[col].isna().mean()
    print(f"{col}: {missing:.2%} missing")

client_id: 0.00% missing
path: 0.00% missing
sentence_id: 0.00% missing
sentence: 0.00% missing
age: 84.31% missing
gender: 86.05% missing
accents: 80.88% missing


In [4]:
# Clip duration csv:
df_duration = df_duration.rename(columns={'clip': 'path', 'duration[ms]': 'duration'}) # change column names to match test split naming convention
# Test split csv:
print(df_test.shape)
print(df_test.columns)

# Drop NaN values:
df_test = df_test.dropna(subset=['client_id', 'path', 'sentence_id', 'sentence'])
print(df_test.shape)

# Remove whitespaces, if any:
df_test['path'] = df_test['path'].str.strip()

# Drop some columns:
df_test = df_test.drop(['sentence_domain', 'up_votes', 'down_votes', 'variant', 'locale', 'segment'], axis=1)

# Add unique id to each row:
df_test['id'] = [str(uuid.uuid4()) for _ in range(len(df_test))]
print(df_test.shape)

# Add clip duration column:
df_test = df_test.merge(df_duration, on='path')
print(df_test.shape)

# Write to metdata.csv file:
df_test.to_csv('metadata.csv', index=False)

(16398, 13)
Index(['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain',
       'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant',
       'locale', 'segment'],
      dtype='str')
(16398, 13)
(16398, 8)
(16398, 9)


In [5]:
# Calculate total duration for the test split:
milliseconds = df_test['duration'].sum()

total_seconds = milliseconds // 1000
hours = total_seconds // 3600
minutes = (total_seconds % 3600) // 60
seconds = total_seconds % 60
time = '%02d:%02d:%02d'%(hours, minutes, seconds)
print(time)

27:09:57


In [7]:
# Filter the audio files in clips folder to contain the ones only present in the test split:
local_path = '/root/master_thesis/thesis_multi_speaker_asr/data/clips'
data_path = '/root/master_thesis/thesis_multi_speaker_asr/data/commonvoice-en/cv-corpus-25.0-2026-03-09/en/clips'
clips = os.listdir(data_path)
print(f'Number of clips in folder: {len(clips)}')

test_files = set(df_test['path'])

for item in clips:
    if item in test_files:
        shutil.copy(
            src=os.path.join(data_path, item),
            dst=local_path
        )


clips_test = os.listdir(local_path)
print(f'Number of clips in new folder: {len(clips_test)}')

Number of clips in folder: 2574404
Number of clips in new folder: 16398


In [7]:
df = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/CORAAL/ATL_metadata_2020.05.txt', sep="\t")
df.to_csv('/root/master_thesis/thesis_multi_speaker_asr/data/CORAAL/metadata.csv', index=False)
df.head()

,CORAAL.Sub,Version.Created,Version.Modified,CORAAL.Spkr,CORAAL.File,Audio.Folder,Tarball,Primary.Spkr,SLAAP.Collection,SLAAP.Spkr,...,Bit.Rate,Sampling.Rate,Source.Device,Dig.Sampling.Rate,Dig.Bit.Rate,Dig.Channels,CORAAL.Length.of.Transcript,CORAAL.Word.Count,Is.Misc.Tier,Notes
0,ATL,v.2020.05,NaN,ATL_se0_ag2_f_01,ATL_se0_ag2_f_01_1,ATL_wav_part03,ATL_audio_part03_2020.05.tar.gz,yes,atl,atl001,...,16 bit,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2339.8,6881,NaN,NaN
1,ATL,v.2020.05,NaN,ATL_se0_ag2_m_01,ATL_se0_ag2_m_01_1,ATL_wav_part04,ATL_audio_part04_2020.05.tar.gz,yes,atl,atl002,...,16 bit,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2440.6,7166,NaN,NaN
2,ATL,v.2020.05,NaN,ATL_se0_ag2_f_02,ATL_se0_ag2_f_02_1,ATL_wav_part03,ATL_audio_part03_2020.05.tar.gz,yes,atl,atl003,...,16 bit,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2497.4,5252,yes,NaN
3,ATL,v.2020.05,NaN,ATL_se0_ag1_m_01,ATL_se0_ag1_m_01_1,ATL_wav_part01,ATL_audio_part01_2020.05.tar.gz,yes,atl,atl004,...,16 bit,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2755.3,7223,NaN,NaN
4,ATL,v.2020.05,NaN,ATL_se0_ag1_f_01,ATL_se0_ag1_f_01_1,ATL_wav_part01,ATL_audio_part01_2020.05.tar.gz,yes,atl,atl005,...,16 bit,44.1 kHz,NaN,44.1 khz,16 bit,Mono,1862.0,4586,NaN,NaN


In [13]:
print(df['CORAAL.File'].iloc[0])

ATL_se0_ag2_f_01_1


In [29]:
import librosa

audio, sr = librosa.load(path='/root/master_thesis/thesis_multi_speaker_asr/data/CORAAL/ATL_se0_ag2_m_03_1.wav')
print(librosa.get_duration(y=audio, sr=sr))


df = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/CORAAL/metadata.csv')
df_2 = pd.DataFrame()
paths = []
ids = []
files = []
for i, row in df.iterrows():
    file = row['CORAAL.File']
    audio_path = file + '.wav'
    paths.append(audio_path)
    files.append(file)
    id = str(uuid.uuid4())
    ids.append(id)

df_2['CORAAL.File'] = files
df_2['path'] = paths
df_2['id'] = ids
df_2.head()

2889.499909297052


,CORAAL.File,path,id
0,ATL_se0_ag2_f_01_1,ATL_se0_ag2_f_01_1.wav,ccfe847f-dff7-41b1-96da-db13f229df73
1,ATL_se0_ag2_m_01_1,ATL_se0_ag2_m_01_1.wav,d3121dd6-4ab1-4fbb-b680-163196558ea6
2,ATL_se0_ag2_f_02_1,ATL_se0_ag2_f_02_1.wav,a9348157-e506-4da8-94b6-615ba3096a14
3,ATL_se0_ag1_m_01_1,ATL_se0_ag1_m_01_1.wav,6be8e426-4757-4672-ae02-76ab69cc7caf
4,ATL_se0_ag1_f_01_1,ATL_se0_ag1_f_01_1.wav,3fe519d3-5a20-4b95-8459-b72fef046216


In [30]:
merged = df.merge(df_2, on='CORAAL.File')
merged.head()

,CORAAL.Sub,Version.Created,Version.Modified,CORAAL.Spkr,CORAAL.File,Audio.Folder,Tarball,Primary.Spkr,SLAAP.Collection,SLAAP.Spkr,...,Source.Device,Dig.Sampling.Rate,Dig.Bit.Rate,Dig.Channels,CORAAL.Length.of.Transcript,CORAAL.Word.Count,Is.Misc.Tier,Notes,path,id
0,ATL,v.2020.05,NaN,ATL_se0_ag2_f_01,ATL_se0_ag2_f_01_1,ATL_wav_part03,ATL_audio_part03_2020.05.tar.gz,yes,atl,atl001,...,NaN,44.1 khz,16 bit,Mono,2339.8,6881,NaN,NaN,ATL_se0_ag2_f_01_1.wav,ccfe847f-dff7-41b1-96da-db13f229df73
1,ATL,v.2020.05,NaN,ATL_se0_ag2_m_01,ATL_se0_ag2_m_01_1,ATL_wav_part04,ATL_audio_part04_2020.05.tar.gz,yes,atl,atl002,...,NaN,44.1 khz,16 bit,Mono,2440.6,7166,NaN,NaN,ATL_se0_ag2_m_01_1.wav,d3121dd6-4ab1-4fbb-b680-163196558ea6
2,ATL,v.2020.05,NaN,ATL_se0_ag2_f_02,ATL_se0_ag2_f_02_1,ATL_wav_part03,ATL_audio_part03_2020.05.tar.gz,yes,atl,atl003,...,NaN,44.1 khz,16 bit,Mono,2497.4,5252,yes,NaN,ATL_se0_ag2_f_02_1.wav,a9348157-e506-4da8-94b6-615ba3096a14
3,ATL,v.2020.05,NaN,ATL_se0_ag1_m_01,ATL_se0_ag1_m_01_1,ATL_wav_part01,ATL_audio_part01_2020.05.tar.gz,yes,atl,atl004,...,NaN,44.1 khz,16 bit,Mono,2755.3,7223,NaN,NaN,ATL_se0_ag1_m_01_1.wav,6be8e426-4757-4672-ae02-76ab69cc7caf
4,ATL,v.2020.05,NaN,ATL_se0_ag1_f_01,ATL_se0_ag1_f_01_1,ATL_wav_part01,ATL_audio_part01_2020.05.tar.gz,yes,atl,atl005,...,NaN,44.1 khz,16 bit,Mono,1862.0,4586,NaN,NaN,ATL_se0_ag1_f_01_1.wav,3fe519d3-5a20-4b95-8459-b72fef046216


# AMI Corpus:
Preprocess the subset of the AMI Meeting Corpus that I plan to use.

In [41]:
import pandas as pd
import numpy as np
import os
import json
import xml.etree.ElementTree as ET
from pathlib import Path
import re
from pydub import AudioSegment
import librosa

NITE = "http://nite.sourceforge.net/" # Global attribute used for ID in all xml files...

In [ ]:
resource_path = os.path.join('../../', 'data/amicorpus/metadata/corpusResources')

# meetings.xml:
meetings_root = ET.parse(os.path.join(resource_path, 'meetings.xml')).getroot()


speaker_rows = []

for meeting in meetings_root.findall(".//meeting"):
    meeting_id = meeting.attrib["observation"]

    for speaker in meeting.findall(".//speaker"):
        speaker_rows.append({
            "meeting_id": meeting_id
        })

speaker_map = pd.DataFrame(speaker_rows)

# participants.xml:
participants_root = ET.parse(os.path.join(resource_path, 'participants.xml')).getroot()

participants = []
for participant in participants_root.findall(".//participant"):
    participants.append(participant.attrib)

participants_df = pd.DataFrame(participants)
participants_df = participants_df.rename(columns={'{http://nite.sourceforge.net/}id': 'participant_id'})

In [43]:
print(speaker_map.shape)
speaker_map.head()

(682, 1)


,meeting_id
0,IS1000a
1,IS1000a
2,IS1000a
3,IS1000a
4,IS1000b


In [4]:
print(participants_df.shape)
participants_df.head()

(189, 5)


,participant_id,sex,age_at_collection,native_language,meeting
0,FEE005,F,20.0,English,ES2002
1,MEE006,M,25.0,English,ES2002
2,MEE007,M,21.0,English,ES2002
3,MEE008,M,27.0,English,ES2002
4,MEE009,M,NaN,English,ES2003


In [5]:
meeting_df = speaker_map.merge(
    participants_df,
    on='participant_id',
    how="left"
)

In [32]:
print(meeting_df.shape)
meeting_df.head(20)
meeting_df.to_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/meeting_metadata.csv', index=False)

(682, 8)


In [52]:
# Copy audio paths into the meeting_metadata.csv: 
audio_dir = os.path.join('../..', 'data/amicorpus/wav')
path_metadata3 = '/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata_3.csv'
df = pd.read_csv(path_metadata3)

meeting_wav_map = {} # maps meetingID to the audio file path
for root, dirs, files in os.walk(audio_dir):
    print(f'Directories list: {dirs}')
    print(f'Files list: {files}')
    if not files:
        continue
    temp = files[0].split(sep='.', maxsplit=1)[0]
    print(f'File: {temp}')
    newPath = os.path.join(temp, files[0])
    print(f'New path: {newPath}')
    meeting_wav_map[temp] = newPath


df['path'] = ''
for i, row in df.iterrows():
    meeting_id = row['meeting_id']
    if meeting_id in meeting_wav_map.keys():
        df.at[i, 'path'] = meeting_wav_map[meeting_id]

df.to_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata_4.csv', index=False)
df.head(20)

Directories list: ['ES2016b', 'ES2016a', 'ES2014b', 'ES2014d', 'ES2014a', 'ES2013c', 'ES2013d', 'ES2016d', 'ES2013a', 'ES2015d', 'ES2015a', 'ES2015c', 'ES2015b', 'ES2013b', 'ES2016c', 'ES2014c']
Files list: []
Directories list: []
Files list: ['ES2016b.Mix-Headset.wav']
File: ES2016b
New path: ES2016b/ES2016b.Mix-Headset.wav
Directories list: []
Files list: ['ES2016a.Mix-Headset.wav']
File: ES2016a
New path: ES2016a/ES2016a.Mix-Headset.wav
Directories list: []
Files list: ['ES2014b.Mix-Headset.wav']
File: ES2014b
New path: ES2014b/ES2014b.Mix-Headset.wav
Directories list: []
Files list: ['ES2014d.Mix-Headset.wav']
File: ES2014d
New path: ES2014d/ES2014d.Mix-Headset.wav
Directories list: []
Files list: ['ES2014a.Mix-Headset.wav']
File: ES2014a
New path: ES2014a/ES2014a.Mix-Headset.wav
Directories list: []
Files list: ['ES2013c.Mix-Headset.wav']
File: ES2013c
New path: ES2013c/ES2013c.Mix-Headset.wav
Directories list: []
Files list: ['ES2013d.Mix-Headset.wav']
File: ES2013d
New path: ES2

,meeting_id,text,path
0,ES2014a,"Right , sostart of the first meeting . Uh .Mm-...",ES2014a/ES2014a.Mix-Headset.wav
1,ES2014b,Right uh .So um .So where's the PowerPoint pre...,ES2014b/ES2014b.Mix-Headset.wav
2,ES2014c,Okay .Right .Conceptual design meeting . Right...,ES2014c/ES2014c.Mix-Headset.wav
3,ES2014d,"So is Why not save that .No , you'll ha have t...",ES2014d/ES2014d.Mix-Headset.wav
4,ES2015a,"Alright , that did nothing . Okay . Welcome to...",ES2015a/ES2015a.Mix-Headset.wav
5,ES2015b,Okay .Yeah . That's okay . That's okay .Okay ....,ES2015b/ES2015b.Mix-Headset.wav
6,ES2015c,Is everyone ready to start ? Okay . Great . We...,ES2015c/ES2015c.Mix-Headset.wav
7,ES2015d,"Okay . Here we go .Alright , the agenda for th...",ES2015d/ES2015d.Mix-Headset.wav
8,ES2016a,"Okay . Oh , that's not gonna work .Oh , alrigh...",ES2016a/ES2016a.Mix-Headset.wav
9,ES2016b,Oh .DuOkay .Hm .Thanks for coming to this meet...,ES2016b/ES2016b.Mix-Headset.wav


In [7]:
rows = []
words_path = os.path.join('../../', 'data/amicorpus/metadata/words')

words = {}
word_order = []


for xml_file in Path(words_path).glob("ES201[4-6]*.words.xml"):
    root = ET.parse(xml_file).getroot()

    meeting_id = xml_file.stem.split(".")[0]
    channel = xml_file.stem.split(".")[1]

    for word in root:
        w_id = word.attrib.get(f"{{{NITE}}}id")
        words[w_id] = {
            "start": float(word.attrib.get("starttime", "nan")),
            "end": float(word.attrib.get("endtime", "nan")),
            "word": word.text if word.text else 'NaN'
        }

        rows.append({
            "meeting_id": meeting_id,
            "channel": channel,
            "word_id": w_id,
            "start": float(word.attrib.get("starttime", "nan")),
            "end": float(word.attrib.get("endtime", "nan")),
            "word": word.text
        })

        word_order.append(w_id)

word_pos = {wid: i for i, wid in enumerate(word_order)}
transcripts_df = pd.DataFrame(rows)

In [11]:
rows = []
words_path = os.path.join('../../', 'data/amicorpus/metadata/segments')
utterances = []

for xml_file in Path(words_path).glob("ES201[4-6]*.segments.xml"):
    seg_root = ET.parse(xml_file).getroot()

    for seg in seg_root.findall('.//segment'):
        child = seg.find('.//{*}child')
        if child is None:
            continue

        ref = child.attrib['href']
        
        info = ref.split('#')[0]
        meeting_id = info.split('.')[0]
        channel = info.split('.')[1]

        word_ids = ref.split('#')[1]
        id_count = word_ids.count('id')
        print(id_count)
        matches = None
        if id_count > 1:
            # A range of words
            matches = re.search(r'id\(([^)]+)\)\.\.id\(([^)]+)\)', word_ids)
        else:
            # A single word
            matches = re.search(r'id\(([^)]+)\)', word_ids)

        word_group = [group for group in matches.groups()]
        utter_words = []
        if len(word_group) > 1:
            start = word_pos[word_group[0]]
            end = word_pos[word_group[1]]
            utter_words = word_order[start:end + 1]
            print(f'Start: {start}, End: {end}, Words: {utter_words}')
        else:
            start = word_pos[word_group[0]]
            utter_words = word_order[start:start + 1]
            print(f'Start: {start}, Words: {utter_words}')
        
        segment_utterance = ' '.join(words[id]['word'] for id in utter_words if words[id] and words[id]['word'] is not 'NaN')

        utterances.append({
        "segment_id": seg.attrib.get(f"{{{NITE}}}id"),
        "start": float(seg.attrib["transcriber_start"]),
        "end": float(seg.attrib["transcriber_end"]),
        "text": segment_utterance,
        'meeting_id': meeting_id,
        'channel': channel
    })

utterances_df = pd.DataFrame(utterances)


<>:42: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:42: SyntaxWarning: "is not" with a literal. Did you mean "!="?
/tmp/ipykernel_50749/2004352216.py:42: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  segment_utterance = ' '.join(words[id]['word'] for id in utter_words if words[id] and words[id]['word'] is not 'NaN')


2
Start: 20022, End: 20037, Words: ['ES2015a.D.words0', 'ES2015a.D.words1', 'ES2015a.D.words2', 'ES2015a.D.words3', 'ES2015a.D.words4', 'ES2015a.D.words5', 'ES2015a.D.words6', 'ES2015a.D.words7', 'ES2015a.D.words8', 'ES2015a.D.words9', 'ES2015a.D.words10', 'ES2015a.D.words11', 'ES2015a.D.words12', 'ES2015a.D.words13', 'ES2015a.D.words14', 'ES2015a.D.words15']
1
Start: 20038, Words: ['ES2015a.D.words16']
1
Start: 20039, Words: ['ES2015a.D.words17']
1
Start: 20040, Words: ['ES2015a.D.words18']
1
Start: 20041, Words: ['ES2015a.D.words19']
2
Start: 20042, End: 20094, Words: ['ES2015a.D.words20', 'ES2015a.D.words21', 'ES2015a.D.words22', 'ES2015a.D.words23', 'ES2015a.D.words24', 'ES2015a.D.words25', 'ES2015a.D.words26', 'ES2015a.D.words27', 'ES2015a.D.words28', 'ES2015a.D.words29', 'ES2015a.D.words30', 'ES2015a.D.words31', 'ES2015a.D.words32', 'ES2015a.D.words33', 'ES2015a.D.words34', 'ES2015a.D.words35', 'ES2015a.D.words36', 'ES2015a.D.words37', 'ES2015a.D.words38', 'ES2015a.D.words39', 'E

In [49]:
transcripts_df = transcripts_df.merge(
    speaker_map,
    on=["meeting_id", "channel"],
    how="left"
)

transcripts_df = transcripts_df.merge(
    participants_df,
    on='participant_id',
    how="left"
)

In [52]:
transcripts_df.to_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata.csv', index=False)

In [12]:
utterances_df = utterances_df.merge(
    speaker_map,
    on=["meeting_id", "channel"],
    how="left"
)

utterances_df = utterances_df.merge(
    participants_df,
    on='participant_id',
    how="left"
)

In [13]:
utterances_df.to_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata2.csv', index=False)

In [24]:
path = '../../data/amicorpus'
amiDir = os.listdir(path=path)
print(amiDir)
durMap = {

}

for meet in amiDir:
    if meet == 'metadata':
        continue
    p = os.path.join(path, meet, 'audio')
    wavs = os.listdir(p)[0]
    audio, sr = librosa.load(os.path.join(p, wavs))
    dur = librosa.get_duration(y=audio, sr=sr)

    durMap[meet] = dur

total_dur = sum(durMap.values())
avg_dur = total_dur / len(durMap.keys())

print(total_dur)
print(durMap)
print(avg_dur)

['ES2016b', 'ES2016a', 'ES2014b', 'ES2014d', 'ES2014a', 'ES2013c', 'metadata', 'ES2013d', 'ES2016d', 'ES2013a', 'ES2015d', 'ES2015a', 'ES2015c', 'ES2015b', 'ES2013b', 'ES2016c', 'ES2014c']
31001.213968253967
{'ES2016b': 2412.224036281179, 'ES2016a': 1384.189387755102, 'ES2014b': 2319.8933786848074, 'ES2014d': 2911.36, 'ES2014a': 1149.0133786848073, 'ES2013c': 2358.1227210884354, 'ES2013d': 1898.56, 'ES2016d': 1523.2533786848073, 'ES2013a': 825.1733786848073, 'ES2015d': 1931.5627210884354, 'ES2015a': 1146.56, 'ES2015c': 2135.8933786848074, 'ES2015b': 2294.85873015873, 'ES2013b': 2128.629387755102, 'ES2016c': 2308.394693877551, 'ES2014c': 2273.525396825397}
1937.575873015873


In [25]:
total_seconds = avg_dur
hours = total_seconds // 3600
minutes = (total_seconds % 3600) // 60
seconds = total_seconds % 60
time = '%02d:%02d:%02d'%(hours, minutes, seconds)
print(time)

00:32:17


In [26]:
total_seconds = total_dur
hours = total_seconds // 3600
minutes = (total_seconds % 3600) // 60
seconds = total_seconds % 60
time = '%02d:%02d:%02d'%(hours, minutes, seconds)
print(time)

08:36:41


# Create the full metadata file for AMI Meeting Corpus
It should contain the entire meeting transcript in a single string. So concatenate the individual segments from the transcript metadata csv and add to the meeting metadata csv file at the meeting_id row.

In [44]:
speaker_map = speaker_map.drop_duplicates(subset=['meeting_id'], ignore_index=True)

In [47]:
print(speaker_map)

    meeting_id
0      IS1000a
1      IS1000b
2      IS1000c
3      IS1000d
4      IS1001a
..         ...
166    EN2006a
167    EN2006b
168    EN2009b
169    EN2009c
170    EN2009d

[171 rows x 1 columns]


In [49]:
segment_df = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata.csv')
meet_df = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/meeting_metadata.csv')

segment_df = segment_df.groupby('meeting_id').apply(lambda x: x.sort_values('start'))
segment_df = segment_df.groupby('meeting_id')['text'].apply(lambda x: x.sum()).reset_index()
print(segment_df)

final_df = speaker_map.merge(
    segment_df,
    how='right',
    on='meeting_id'
)
final_df.head(20)


   meeting_id                                               text
0     ES2014a  Right , sostart of the first meeting . Uh .Mm-...
1     ES2014b  Right uh .So um .So where's the PowerPoint pre...
2     ES2014c  Okay .Right .Conceptual design meeting . Right...
3     ES2014d  So is Why not save that .No , you'll ha have t...
4     ES2015a  Alright , that did nothing . Okay . Welcome to...
5     ES2015b  Okay .Yeah . That's okay . That's okay .Okay ....
6     ES2015c  Is everyone ready to start ? Okay . Great . We...
7     ES2015d  Okay . Here we go .Alright , the agenda for th...
8     ES2016a  Okay . Oh , that's not gonna work .Oh , alrigh...
9     ES2016b  Oh .DuOkay .Hm .Thanks for coming to this meet...
10    ES2016c  Okay ..Right .Okay .Alright . Is everyone here...
11    ES2016d  Yep . Soon as Iget this .Okay . This is our la...


,meeting_id,text
0,ES2014a,"Right , sostart of the first meeting . Uh .Mm-..."
1,ES2014b,Right uh .So um .So where's the PowerPoint pre...
2,ES2014c,Okay .Right .Conceptual design meeting . Right...
3,ES2014d,"So is Why not save that .No , you'll ha have t..."
4,ES2015a,"Alright , that did nothing . Okay . Welcome to..."
5,ES2015b,Okay .Yeah . That's okay . That's okay .Okay ....
6,ES2015c,Is everyone ready to start ? Okay . Great . We...
7,ES2015d,"Okay . Here we go .Alright , the agenda for th..."
8,ES2016a,"Okay . Oh , that's not gonna work .Oh , alrigh..."
9,ES2016b,Oh .DuOkay .Hm .Thanks for coming to this meet...


In [50]:
final_df.to_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata_3.csv', index=False)